In [0]:
# Databricks Notebook
# 03_gold_transform

from pyspark.sql.functions import (
    col,
    row_number,
    dense_rank,
    avg,
    max,
    min,
    count,
    round,
    when,
    hour,
    date_format
)

from pyspark.sql.window import Window

# ============================================================
# Configuration
# ============================================================

SILVER_FACILITY_TABLE = "smart_carparking.silver_carpark_occupancy"

GOLD_CURRENT_STATUS_TABLE = "smart_carparking.gold_current_facility_status"
GOLD_HOURLY_TRENDS_TABLE = "smart_carparking.gold_hourly_occupancy_trends"
GOLD_FACILITY_RANKING_TABLE = "smart_carparking.gold_facility_occupancy_ranking"
GOLD_CONGESTION_TIMELINE_TABLE = "smart_carparking.gold_congestion_timeline"

# ============================================================
# Read Silver Facility table
# ============================================================

silver_facility_df = spark.table(SILVER_FACILITY_TABLE)

# ============================================================
# Basic filtering for Gold layer
# Gold should only use valid, analytics-ready records
# ============================================================

valid_silver_df = (
    silver_facility_df
    .filter(col("data_quality_status") == "Valid")
    .filter(col("facility_id").isNotNull())
    .filter(col("message_datetime").isNotNull())
    .filter(col("occupancy_rate").isNotNull())
)

# ============================================================
# GOLD 1 - Current Facility Status
# Latest snapshot per facility
# ============================================================

# Window function:
# For each facility, order records from newest to oldest.
latest_facility_window = (
    Window
    .partitionBy("facility_id")
    .orderBy(col("message_datetime").desc(), col("ingestion_timestamp").desc())
)

latest_facility_df = (
    valid_silver_df
    .withColumn("row_num", row_number().over(latest_facility_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

gold_current_df = (
    latest_facility_df

    # Business rule: parking status based on current occupancy
    .withColumn(
        "parking_status",
        when(col("occupancy_rate") >= 95, "Full")
        .when(col("occupancy_rate") >= 85, "Almost Full")
        .when(col("occupancy_rate") >= 70, "Busy")
        .otherwise("Available")
    )

    # Business rule: pressure level for dashboard colour/category
    .withColumn(
        "pressure_level",
        when(col("occupancy_rate") >= 90, "Critical")
        .when(col("occupancy_rate") >= 75, "High")
        .when(col("occupancy_rate") >= 50, "Medium")
        .otherwise("Low")
    )

    # Business recommendation for AI/dashboard text
    .withColumn(
        "recommendation",
        when(
            col("occupancy_rate") >= 95,
            "This facility is currently full."
        )
        .when(
            col("occupancy_rate") >= 85,
            "Limited parking availability."
        )
        .when(
            col("occupancy_rate") >= 70,
            "Parking demand is increasing."
        )
        .otherwise(
            "Parking availability is currently good."
        )
    )
    .orderBy(col("occupancy_rate").desc())
)

gold_current_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_CURRENT_STATUS_TABLE)

display(gold_current_df)

# ============================================================
# GOLD 2 - Hourly Occupancy Trends
# Historical occupancy analytics by facility and hour
# ============================================================

gold_hourly_trends_df = (
    valid_silver_df

    # Extract hour from timestamp
    .withColumn("hour_of_day", hour(col("message_datetime")))

    # Aggregate historical snapshots by facility and hour
    .groupBy(
        "facility_id",
        "facility_name",
        "suburb",
        "hour_of_day"
    )
    .agg(
        round(avg("occupancy_rate"), 2).alias("avg_occupancy_rate"),
        round(avg("available_spaces"), 2).alias("avg_available_spaces"),
        round(avg("occupied_spaces"), 2).alias("avg_occupied_spaces"),
        count("*").alias("snapshot_count")
    )

    # Categorise historical congestion pressure
    .withColumn(
        "peak_pressure_level",
        when(col("avg_occupancy_rate") >= 90, "Critical")
        .when(col("avg_occupancy_rate") >= 75, "High")
        .when(col("avg_occupancy_rate") >= 50, "Medium")
        .otherwise("Low")
    )
    .orderBy("facility_name", "hour_of_day")
)

gold_hourly_trends_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_HOURLY_TRENDS_TABLE)

display(gold_hourly_trends_df)

# ============================================================
# GOLD 3 - Facility Occupancy Ranking
# Ranks facilities by average historical occupancy
# ============================================================

facility_ranking_df = (
    valid_silver_df
    .groupBy(
        "facility_id",
        "facility_name",
        "suburb"
    )
    .agg(
        round(avg("occupancy_rate"), 2).alias("avg_occupancy_rate"),
        round(max("occupancy_rate"), 2).alias("max_occupancy_rate"),
        round(min("occupancy_rate"), 2).alias("min_occupancy_rate"),
        count("*").alias("snapshot_count")
    )
)

ranking_window = Window.orderBy(col("avg_occupancy_rate").desc())

facility_ranking_df = (
    facility_ranking_df
    .withColumn("occupancy_rank", dense_rank().over(ranking_window))
    .withColumn(
        "demand_category",
        when(col("avg_occupancy_rate") >= 90, "Consistently Critical")
        .when(col("avg_occupancy_rate") >= 75, "High Demand")
        .when(col("avg_occupancy_rate") >= 50, "Moderate Demand")
        .otherwise("Low Demand")
    )
    .orderBy("occupancy_rank")
)

facility_ranking_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_FACILITY_RANKING_TABLE)

display(facility_ranking_df)

# ============================================================
# GOLD 4 - Congestion Timeline
# Designed for heatmap visualisation
# Helps answer: "When should the city intervene?"
# ============================================================

congestion_timeline_df = (
    valid_silver_df

    # Extract day name and hour for timeline analytics
    .withColumn("day_of_week", date_format(col("message_datetime"), "EEEE"))
    .withColumn("hour_of_day", hour(col("message_datetime")))

    .groupBy(
        "suburb",
        "day_of_week",
        "hour_of_day"
    )
    .agg(
        round(avg("occupancy_rate"), 2).alias("avg_occupancy_rate"),
        round(avg("available_spaces"), 2).alias("avg_available_spaces"),
        count("*").alias("snapshot_count")
    )

    # Risk level for dashboard heatmap
    .withColumn(
        "congestion_risk",
        when(col("avg_occupancy_rate") >= 90, "Critical")
        .when(col("avg_occupancy_rate") >= 75, "High")
        .when(col("avg_occupancy_rate") >= 50, "Medium")
        .otherwise("Low")
    )

    # Simple business explanation
    .withColumn(
        "insight",
        when(
            col("avg_occupancy_rate") >= 90,
            "Very high parking pressure. Intervention may be required."
        )
        .when(
            col("avg_occupancy_rate") >= 75,
            "High parking demand. Monitor closely during this period."
        )
        .when(
            col("avg_occupancy_rate") >= 50,
            "Moderate demand. Parking is still available but usage is increasing."
        )
        .otherwise(
            "Low congestion risk. Parking availability is generally good."
        )
    )
    .orderBy("suburb", "day_of_week", "hour_of_day")
)

congestion_timeline_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_CONGESTION_TIMELINE_TABLE)

display(congestion_timeline_df)

facility_id,facility_name,tfnsw_facility_id,tsn,api_time_seconds_since_2000,park_id,suburb,address,latitude,longitude,total_spots,occupied_spaces,loop_count,transient_vehicles,monthly_vehicles,open_gate_count,message_datetime,ingestion_timestamp,source_system,available_spaces,occupancy_rate,data_quality_status,facility_snapshot_key,parking_status,pressure_level,recommendation
20,Campbelltown Hurley St,256020TPR002,256020,832471242,1,Campbelltown,Hurley Street,-34.065798,150.812432,113,113,442335,null,null,null,2026-05-19T12:00:42.000Z,2026-05-19T02:06:46.675Z,TfNSW Car Park API,0,100.0,Valid,e7deae07a28e2db7a60cd9db6f9949a61f6061454d3d455a9d298bc9023cadf3,Full,Critical,This facility is currently full.
33,Cherrybrook,2126158TPR001,2126158,832471545,1,Cherrybrook,Bradfield Parade,-33.737374,151.033431,384,384,307519,null,null,null,2026-05-19T12:05:45.000Z,2026-05-19T02:06:46.675Z,TfNSW Car Park API,0,100.0,Valid,2dab339b49084e7a242b777339c57de15aaaa5d42c57ff73f4de4cef7b479bf7,Full,Critical,This facility is currently full.
19,Campbelltown Farrow Rd (north),256020TPR001,256020,832471266,1,Campbelltown,Farrow Road,-34.062279,150.815283,68,68,384077,null,null,null,2026-05-19T12:01:06.000Z,2026-05-19T02:06:46.675Z,TfNSW Car Park API,0,100.0,Valid,9ff9c4690679d4445644fa6a930d3612003bff79ded1fab048229caa973b7f01,Full,Critical,This facility is currently full.
32,Hills Showground,2154392TPR001,2154392,832471410,1,Castle Hill,De Clambe Drive,-33.727735,150.98505,584,584,93035,null,null,null,2026-05-19T12:03:30.000Z,2026-05-19T02:06:46.675Z,TfNSW Car Park API,0,100.0,Valid,96566f0e601141b940a0c13298341c61e33590480d9ef989421606d55e319e93,Full,Critical,This facility is currently full.
26,Tallawong P1,2155384TPR001,2155384,832471358,1,Tallawong,Conferta Avenue,-33.69304704,150.9052577,123,123,7004,null,null,null,2026-05-19T12:02:38.000Z,2026-05-19T02:06:46.675Z,TfNSW Car Park API,0,100.0,Valid,4302786ad2f8baeacc4bfb6096c09ca2d2789586621928218ab6601b4d6fbdda,Full,Critical,This facility is currently full.
14,West Ryde,211420TPR001,211420,832471554,1,West Ryde,Ryedale Road,-33.805993,151.091248,151,151,524008,null,null,null,2026-05-19T12:05:54.000Z,2026-05-19T02:06:46.675Z,TfNSW Car Park API,0,100.0,Valid,d11cd42dc1b2209f81b8f765ffbe3b5c32570f0d8a3f36a8e47dbc8d67d7e1a0,Full,Critical,This facility is currently full.
34,Lindfield Village Green,207010TPR001,207010,832471471,1,Lindfield,Tryon Road,-33.77449,151.170549,94,94,18167,null,null,null,2026-05-19T12:04:31.000Z,2026-05-19T02:06:46.675Z,TfNSW Car Park API,0,100.0,Valid,e456210967d1dbbcc8c61c303791a590c5fa095d08309ec64d5d66183407ff94,Full,Critical,This facility is currently full.
30,Kellyville (south),2155382TPR002,2155382,832471493,1,Kellyville,Guragura Street,-33.71498982,150.9363451,964,964,314347,null,null,null,2026-05-19T12:04:53.000Z,2026-05-19T02:06:46.675Z,TfNSW Car Park API,0,100.0,Valid,0d1bc6636089884e9ad113bb818335b23fceaf9745ab2bc155ed4e3b66b86005,Full,Critical,This facility is currently full.
29,Kellyville (north),2155382TPR001,2155382,832471505,1,Kellyville,Derrobarry Street,-33.711156,150.934364,351,351,521896,null,null,null,2026-05-19T12:05:05.000Z,2026-05-19T02:06:46.675Z,TfNSW Car Park API,0,100.0,Valid,4978b61c7a895afc051788360e656d85214660f718450353b8c2e1129342ebee,Full,Critical,This facility is currently full.
31,Bella Vista,2153478TPR001,2153478,832471535,1,Bella Vista,Byles Place,-33.727438,150.941761,774,774,239651,null,null,null,2026-05-19T12:05:35.000Z,2026-05-19T02:06:46.675Z,TfNSW Car Park API,0,100.0,Valid,ea1a96a235dd60d5cfec57dde327fa2052ae0b906559afad9a22e64d0577ebab,Full,Critical,This facility is currently full.


facility_id,facility_name,suburb,hour_of_day,avg_occupancy_rate,avg_available_spaces,avg_occupied_spaces,snapshot_count,peak_pressure_level
486,Ashfield,Ashfield,0,10.88,192.5,23.5,2,Low
486,Ashfield,Ashfield,2,2.37,212.83,5.17,12,Low
486,Ashfield,Ashfield,3,4.63,206.0,10.0,1,Low
486,Ashfield,Ashfield,4,5.21,204.75,11.25,4,Low
486,Ashfield,Ashfield,5,7.19,203.25,15.75,8,Low
486,Ashfield,Ashfield,6,12.04,190.0,26.0,1,Low
486,Ashfield,Ashfield,7,39.75,130.17,87.83,6,Low
486,Ashfield,Ashfield,8,16.2,181.0,35.0,1,Low
486,Ashfield,Ashfield,9,25.46,161.0,55.0,1,Low
486,Ashfield,Ashfield,10,19.91,173.0,43.0,1,Low


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


facility_id,facility_name,suburb,avg_occupancy_rate,max_occupancy_rate,min_occupancy_rate,snapshot_count,occupancy_rank,demand_category
19,Campbelltown Farrow Rd (north),Campbelltown,69.86,100.0,20.59,119,1,Moderate Demand
26,Tallawong P1,Tallawong,54.93,100.0,0.0,119,2,Moderate Demand
20,Campbelltown Hurley St,Campbelltown,44.79,100.0,0.0,119,3,Low Demand
33,Cherrybrook,Cherrybrook,43.91,100.0,0.0,119,4,Low Demand
14,West Ryde,West Ryde,39.86,100.0,0.0,119,5,Low Demand
11,Narrabeen,Narrabeen,38.93,100.0,0.0,109,6,Low Demand
25,Hornsby,Hornsby,38.79,100.0,0.69,118,7,Low Demand
31,Bella Vista,Bella Vista,37.19,100.0,0.0,119,8,Low Demand
12,Mona Vale,Mona Vale,36.67,100.0,0.0,98,9,Low Demand
32,Hills Showground,Castle Hill,36.25,100.0,0.0,119,10,Low Demand


suburb,day_of_week,hour_of_day,avg_occupancy_rate,avg_available_spaces,snapshot_count,congestion_risk,insight
Ashfield,Friday,2,0.0,216.0,1,Low,Low congestion risk. Parking availability is generally good.
Ashfield,Friday,5,6.02,203.0,1,Low,Low congestion risk. Parking availability is generally good.
Ashfield,Friday,7,87.04,28.0,1,High,High parking demand. Monitor closely during this period.
Ashfield,Friday,15,90.28,21.0,1,Critical,Very high parking pressure. Intervention may be required.
Ashfield,Friday,16,97.69,5.0,1,Critical,Very high parking pressure. Intervention may be required.
Ashfield,Friday,17,84.03,34.5,2,High,High parking demand. Monitor closely during this period.
Ashfield,Friday,18,78.24,47.0,1,High,High parking demand. Monitor closely during this period.
Ashfield,Friday,19,73.61,57.0,1,Medium,Moderate demand. Parking is still available but usage is increasing.
Ashfield,Friday,20,60.96,84.33,3,Medium,Moderate demand. Parking is still available but usage is increasing.
Ashfield,Friday,22,29.47,152.33,3,Low,Low congestion risk. Parking availability is generally good.
